In [49]:
from embedder import Embedder
embed = Embedder()

query1 = "How does approximate nearest neighbor search work?"
v_query1 = embed.encode(query1)

v_query1[0]

np.float64(-0.02058203437252893)

In [50]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

documents = [file.parse() for file in reader.read()]

In [51]:
text = [doc["content"] for doc in documents if doc["filename"]=='02-vector-search/lessons/07-sqlitesearch-vector.md']

v_text = embed.encode(text[0])

scores = v_text.dot(v_query1)
scores

np.float64(0.36107027225589694)

In [52]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk["content"] for chunk in chunks]

X = []
batch_vectors = embed.encode_batch(texts)
X.extend(batch_vectors)

import numpy as np
X = np.array(X)

scores = X.dot(v_query1)
idx = np.argmax(scores)

chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [53]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

query2 = "What metric do we use to evaluate a search engine?"
v_query2 = embed.encode(query2)
vector_results = vindex.search(v_query2, num_results=1)
vector_results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [54]:
query3 = "How do I store vectors in PostgreSQL?"

from minsearch import Index
index_chunks = Index(
    text_fields=["content"],
)
index_chunks.fit(chunks)

keyword_results = index_chunks.search(query3, num_results=5)
v_query3 = embed.encode(query3)
vector_results = vindex.search(v_query3, num_results=5)

print("Text Search Results: ")
print([i["filename"] for i in keyword_results])
print("Vector Search Results: ")
print([v["filename"] for v in vector_results])

Text Search Results: 
['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
Vector Search Results: 
['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


In [55]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query4 = "How do I give the model access to tools?"
keyword_results = index_chunks.search(query4, num_results=5)
v_query4 = embed.encode(query4)
vector_results = vindex.search(v_query4, num_results=5)

results = rrf([vector_results, keyword_results])
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'